# Step 0 · Cleaning and family-level datasets

Input: the sector built by `01_extract.ipynb` (`outputs/<env>/step0/`).
Output: the clean datasets used from Step 1 on (`outputs/<env>/step1/`).

The PW kick-off asks for: DOCDB family as unit, earliest priority year, harmonised applicants (PSN/HAN plus own
rules for subsidiaries), truncation flag, and the tables `tls201`, `tls224`, `tls206/207`, `tls212`, `tls202/203`.
The lectures add other information worth collecting (*who, what, when, where, how* of IM_Theory_3): inventors,
legal status and renewals, oppositions and transfers, science links (NPL citations), citation categories,
claims, grant lag. Each extra block can be switched off in `config.py` (`FETCH_...`).

| Section | Output (`outputs/<env>/step1/`) | Used for (step · concept) |
|---|---|---|
| 2 Families and applications | `families.csv`, `applications.csv` | Step 1 volumes at both levels · Step 3 success, globalization |
| 3 CPC | `family_cpc.csv`, `family_subclass.csv`, `application_subclass.csv` | Step 1 technology mix (fractional) · Step 2 subfields · Step 4 novel recombination |
| 4 WIPO fields | `family_wipo_fields.csv` | Step 1 technology mix (Schmoch, 2008) |
| 5 Publications | columns in `families.csv` | grant lag, number of claims (Squicciarini et al., 2013) |
| 6 Citations | `citations_forward.csv`, `citations_backward.csv`, `citation_categories.csv`, `cited_citing_cpc.csv` | Step 3 influence · Step 4 originality, generality · science linkage (NPL) |
| 7 Applicants | `applicants.csv`, `applicant_countries_by_application.csv`, `applicants_to_review.csv`, `applicants_audit.csv`, `applicants_profile.csv` | Step 1 geography · Step 2 HHI, entry, top players · Step 6 co-assignments |
| 8 Inventors | `inventors.csv`, `inventor_countries_by_application.csv` | Step 1 geography of R&D · team size, collaboration networks |
| 9 Legal events | `legal_events.csv` + flags in `families.csv` | active families, renewals, oppositions, transfers (IM_Theory_1–2) |
| 10 Benchmarks | `benchmark_families_per_year.csv`, `benchmark_applications_per_year.csv` | Step 1 growth vs all patents and vs a neighbouring field |

**Counting rules** (Tutorial 0, Notebook 2, section 4; state them in the technical appendix):
- **Unit**: the DOCDB family (PW rule); every family-level table also has an application-level twin, so Step 1
  can show both levels side by side as in Tutorial 0.
- **Truncation**: years after `LAST_COMPLETE_YEAR` are flagged (`is_truncated`, `is_truncated_appln`); figures shade
  them, and growth rates must never use them.
- **PCT**: `appln_auth = 'WO'` and `appln_kind` starting with `W`. Family level: at least one PCT application.
- **Fractional counting** (`pw_utils.fractional_weights`): each unit is split equally among its distinct persons
  with a known country (countries), or among its distinct CPC subclasses (technology). Every unit sums to 1.

Family year = earliest priority of the whole family; language is not a filter (only `has_en_abstract` for NLP);
applicants are consolidated into corporate groups with documented rules.

## 1 · Setup

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import and_, func

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS206_PERSON,
    TLS207_PERS_APPLN,
    TLS224_APPLN_CPC,
)

sys.path.append(str(Path.cwd()))
import config as C
from pw_utils import (applicant_type, available_columns, clean_country, clean_patent_text, fetch_by_ids,
                      fractional_weights, get_model, legal_event_flags, normalise_name, to_group)

RAW = C.OUT_STEP0 / "raw"
C.OUT_STEP1.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=C.ENV)
db = patstat.orm()

# optional tables: None if this tipdata version names them differently
TLS211 = get_model("TLS211_PAT_PUBLN")
TLS212 = get_model("TLS212_CITATION")
TLS215 = get_model("TLS215_CITN_CATEG")
TLS228 = get_model("TLS228_DOCDB_FAM_CITN")          # table tls228_docdb_fam_citn (Tutorial 0, schema list)
TLS230 = get_model("TLS230_APPLN_TECHN_FIELD")
TLS231 = get_model("TLS231_INPADOC_LEGAL_EVENT")
TLS803 = get_model("TLS803_LEGAL_EVENT_CODE")
TLS901 = get_model("TLS901_TECHN_FIELD_IPC")
print({n: m is not None for n, m in [("tls211", TLS211), ("tls212", TLS212), ("tls215", TLS215), ("tls228", TLS228),
                                      ("tls230", TLS230), ("tls231", TLS231), ("tls803", TLS803), ("tls901", TLS901)]})


def fetch(make_query, ids, columns=None):
    """Chunked query; returns an empty frame with `columns` when nothing comes back."""
    out = fetch_by_ids(patstat, make_query, ids)
    return pd.DataFrame(columns=columns) if out.empty and columns else out

{'tls211': True, 'tls212': True, 'tls215': True, 'tls228': True, 'tls230': True, 'tls231': True, 'tls803': True, 'tls901': True}


In [2]:
sector = pd.read_csv(C.OUT_STEP0 / "sector_families.csv")
apps = pd.read_csv(RAW / "applications.csv.gz")
titles = pd.read_csv(RAW / "titles.csv.gz")
abstracts = pd.read_csv(RAW / "abstracts.csv.gz")
cpc = pd.read_csv(RAW / "cpc.csv.gz")

assert sector["docdb_family_id"].is_unique, "sector_families.csv must have one row per family"
family_ids = sector["docdb_family_id"].astype("int64").tolist()
appln_ids = apps["appln_id"].astype("int64").unique().tolist()
print(f"env {C.ENV} | IPR {C.IPR_TYPE} | sector families: {len(family_ids):,} | applications: {len(appln_ids):,}")

env PROD | IPR PI | sector families: 11,999 | applications: 16,179


## 2 · Family table

- `family_year`: earliest priority year of the family (minimum over its applications), computed in `01_extract`.
- `is_truncated`: family year after `LAST_COMPLETE_YEAR`; shaded in figures and excluded from growth rates.
- `granted_any`: at least one application of the family was granted (success; only on filings old enough).
- `n_offices`, `n_national_offices`: distinct filing offices, with and without `WO` and `EP` (globalization).
- `is_international`: protection sought beyond the first office (IM_Theory_3, "international family").
- `via_pct`: at least one PCT application (`appln_auth = 'WO'` and `appln_kind` starting with `W`, Tutorial 0).
- `inpadoc_family_id`: extended family (linked through at least one shared priority). The PW uses the DOCDB
  simple family; the INPADOC id is kept to check that results do not change with the wider definition.
- `title_en`, `abstract_en`, `clean_abstract`: English text for NLP only.

In [3]:
offices = apps.groupby("docdb_family_id")["appln_auth"]
apps["is_pct"] = apps["appln_auth"].eq("WO") & apps["appln_kind"].astype(str).str.startswith("W")

families = sector[["docdb_family_id", "family_year", "is_truncated", "first_office", "has_en_abstract",
                   "block_combo"]].merge(
    pd.DataFrame({
        "inpadoc_family_id": apps.groupby("docdb_family_id")["inpadoc_family_id"].first(),
        "n_applications": apps.groupby("docdb_family_id")["appln_id"].nunique(),
        "docdb_family_size": apps.groupby("docdb_family_id")["docdb_family_size"].max(),
        "nb_citing_docdb_fam": apps.groupby("docdb_family_id")["nb_citing_docdb_fam"].max(),
        "granted_any": apps.assign(g=apps["granted"].eq("Y")).groupby("docdb_family_id")["g"].any(),
        "via_pct": apps.groupby("docdb_family_id")["is_pct"].any(),
        "n_offices": offices.nunique(),
        "n_national_offices": offices.agg(lambda s: s[~s.isin(["WO", "EP"])].nunique()),
        "has_ep": offices.agg(lambda s: s.eq("EP").any()),
        "offices": offices.agg(lambda s: " ".join(sorted(s.unique()))),
        "first_publn_year": apps.groupby("docdb_family_id")["earliest_publn_year"].min(),
    }).reset_index(),
    on="docdb_family_id", how="left",
)
families["is_international"] = families["n_offices"] > 1

en_title = (titles[(titles["appln_title_lg"] == "en") & titles["appln_title"].fillna("").str.strip().ne("")]
            .drop_duplicates("docdb_family_id")[["docdb_family_id", "appln_title"]]
            .rename(columns={"appln_title": "title_en"}))
any_title = titles.drop_duplicates("docdb_family_id")[["docdb_family_id", "appln_title"]].rename(
    columns={"appln_title": "title_any"})
en_abs = (abstracts[(abstracts["appln_abstract_lg"] == "en") & abstracts["appln_abstract"].fillna("").str.strip().ne("")]
          .drop_duplicates("docdb_family_id")[["docdb_family_id", "appln_abstract"]]
          .rename(columns={"appln_abstract": "abstract_en"}))
families = (families.merge(en_title, on="docdb_family_id", how="left")
                    .merge(any_title, on="docdb_family_id", how="left")
                    .merge(en_abs, on="docdb_family_id", how="left"))
families["title"] = families["title_en"].fillna(families["title_any"])
families["clean_abstract"] = families["abstract_en"].apply(clean_patent_text)
families = families.drop(columns="title_any")

# application-level twin (Tutorial 0: the sector is selected at family level, then all its applications are counted)
# apps already carries family_year (computed in 01_extract); add only the family truncation flag
applications = apps.merge(families[["docdb_family_id", "is_truncated"]], on="docdb_family_id", how="left")
applications = applications.rename(columns={"is_truncated": "is_truncated_family"})
applications["is_truncated_appln"] = applications["appln_filing_year"] > C.LAST_COMPLETE_YEAR
applications["in_appln_window"] = applications["appln_filing_year"].between(C.YEAR_MIN, C.YEAR_MAX)
applications["is_granted"] = applications["granted"].eq("Y")
print(f"applications: {len(applications):,} | per family: mean {len(applications) / len(families):.2f} "
      f"| outside {C.YEAR_MIN}-{C.YEAR_MAX} by filing year: {(~applications['in_appln_window']).sum():,} "
      f"(kept; filter with in_appln_window for application-level counts, as in Tutorial 0)")
print(f"families: {len(families):,} | English abstract: {families['abstract_en'].notna().sum():,} "
      f"| granted: {families['granted_any'].mean():.1%} | via PCT: {families['via_pct'].mean():.1%} "
      f"| international: {families['is_international'].mean():.1%} | truncated: {families['is_truncated'].sum():,}")

applications: 16,179 | per family: mean 1.35 | outside 2000-2025 by filing year: 23 (kept; filter with in_appln_window for application-level counts, as in Tutorial 0)
families: 11,999 | English abstract: 11,868 | granted: 53.9% | via PCT: 8.6% | international: 9.8% | truncated: 4,198


## 3 · CPC codes per family

One row per family × CPC symbol, with the **subclass** (4 characters, e.g. `B63G`) and the **main group**
(e.g. `B63G8/`). Section `Y` (cross-sectional tagging, e.g. `Y02E`) is flagged: it describes an application
domain, not a technology. Fractional counts (1/k per subclass, Tutorial 0) are computed in Step 1.

In [4]:
family_cpc = cpc[["docdb_family_id", "cpc"]].drop_duplicates().copy()
family_cpc["subclass"] = family_cpc["cpc"].str[:4]
family_cpc["main_group"] = family_cpc["cpc"].str.extract(r"^([A-Z]\d{2}[A-Z]\d+/)", expand=False)
family_cpc["is_tagging"] = family_cpc["cpc"].str.startswith("Y")
families = families.merge(
    family_cpc[~family_cpc["is_tagging"]].groupby("docdb_family_id")["subclass"].nunique().rename("n_subclasses"),
    on="docdb_family_id", how="left")
# fractional counts by subclass: `weight` uses all subclasses (Tutorial 0), `weight_tech` leaves out the Y tagging section
family_subclass = fractional_weights(family_cpc, "docdb_family_id", "subclass")[["docdb_family_id", "subclass", "weight"]]
family_subclass = family_subclass.merge(
    fractional_weights(family_cpc[~family_cpc["is_tagging"]], "docdb_family_id", "subclass")[
        ["docdb_family_id", "subclass", "weight"]].rename(columns={"weight": "weight_tech"}),
    on=["docdb_family_id", "subclass"], how="left")
app_cpc = cpc.assign(subclass=cpc["cpc"].str[:4])
application_subclass = fractional_weights(app_cpc, "appln_id", "subclass")[["appln_id", "docdb_family_id", "subclass", "weight"]]
print(f"family_cpc: {len(family_cpc):,} rows | distinct subclasses: {family_cpc['subclass'].nunique():,}")
display(family_cpc.drop_duplicates(["docdb_family_id", "subclass"])["subclass"].value_counts().head(15)
        .to_frame("families"))

family_cpc: 47,863 rows | distinct subclasses: 294


,families
subclass,
B63G,4022
B63C,3925
B63B,1706
G05D,1314
B63H,1196
B25J,818
G01S,735
Y02A,670
G01C,664


## 4 · WIPO technology fields

35 fields in 5 sectors (Schmoch, 2008), from `tls230` (weights summing to 1 per application) and `tls901`
(names). Weights are rescaled so that every **family** sums to 1, as in Tutorial 0, Notebook 2.

In [5]:
if C.FETCH_WIPO_FIELDS and TLS230 is not None:
    wf = fetch(lambda ch: db.query(TLS201_APPLN.docdb_family_id, TLS230.appln_id, TLS230.techn_field_nr, TLS230.weight)
               .join(TLS230, TLS230.appln_id == TLS201_APPLN.appln_id)
               .filter(TLS201_APPLN.appln_id.in_(ch)), appln_ids)
    if not wf.empty:
        wf[["appln_id", "techn_field_nr", "weight"]].to_csv(C.OUT_STEP1 / "application_wipo_fields.csv", index=False)  # sums to 1 per application
        wf = wf.groupby(["docdb_family_id", "techn_field_nr"], as_index=False)["weight"].sum()
        wf["weight"] = wf["weight"] / wf.groupby("docdb_family_id")["weight"].transform("sum")
        if TLS901 is not None:
            names = pd.DataFrame(patstat.df(db.query(TLS901.techn_field_nr, TLS901.techn_field,
                                                     TLS901.techn_sector).distinct())).drop_duplicates("techn_field_nr")
            wf = wf.merge(names, on="techn_field_nr", how="left")
        wf.to_csv(C.OUT_STEP1 / "family_wipo_fields.csv", index=False)
        if "techn_sector" in wf:
            display(wf.groupby("techn_sector")["weight"].sum().div(wf["weight"].sum()).mul(100).round(1)
                    .sort_values(ascending=False).to_frame("% of families"))

,% of families
techn_sector,
Mechanical engineering,55.5
Instruments,26.8
Electrical engineering,14.2
Other fields,2.2
Chemistry,1.3


## 5 · Publications: grant date and claims

From `tls211_pat_publn`: the first grant publication of each family gives the **grant lag** (years from the
priority to the first grant: examiners take 2–5 years, IM_Theory_1), and `publn_claims` gives the **number of
claims**, a quality indicator of the OECD set (Squicciarini, Dernis & Criscuolo, 2013): more claims, broader
protection. The publication ids are also needed for the citation details of Section 6.

In [6]:
publications = pd.DataFrame()
if C.FETCH_PUBLICATIONS and TLS211 is not None:
    cols = available_columns(TLS211, ["pat_publn_id", "appln_id", "publn_auth", "publn_kind", "publn_date",
                                      "publn_first_grant", "publn_claims"])
    publications = fetch(lambda ch: db.query(*cols).filter(TLS211.appln_id.in_(ch)), appln_ids)
    publications = publications.merge(apps[["appln_id", "docdb_family_id"]], on="appln_id", how="left")
    publications["publn_year"] = pd.to_datetime(publications["publn_date"], errors="coerce").dt.year
    publications.loc[publications["publn_year"] >= C.MISSING_YEAR, "publn_year"] = np.nan

    if "publn_first_grant" in publications:
        grants = publications[publications["publn_first_grant"].astype(str).str.strip().eq("Y")]
        first_grant = grants.groupby("docdb_family_id")["publn_year"].min().rename("first_grant_year").reset_index()
        families = families.merge(first_grant, on="docdb_family_id", how="left")
        families["grant_lag"] = families["first_grant_year"] - families["family_year"]
        print(f"families with a grant publication: {families['first_grant_year'].notna().sum():,} "
              f"| median grant lag: {families['grant_lag'].median()} years")
    if "publn_claims" in publications:
        claims = (pd.to_numeric(publications["publn_claims"], errors="coerce").replace(0, np.nan)
          .groupby(publications["docdb_family_id"]).max().rename("max_claims").reset_index())
        families = families.merge(claims, on="docdb_family_id", how="left")
        print(f"median number of claims: {families['max_claims'].median()}")
    print(f"publications: {len(publications):,}")

families with a grant publication: 6,455 | median grant lag: 2.0 years
median number of claims: 17.0
publications: 24,200


## 6 · Citations

### 6.1 Family-level citation links (`tls228_docdb_fam_citn`)
A citing DOCDB family → a cited DOCDB family, so each invention is counted once even when several publications
of the same family cite the same document.

- **forward** citations: families citing a sector family (influence, Hall, Jaffe & Trajtenberg 2001);
  `within_5y` allows a fixed 5-year window, `fwd_window_complete` marks families whose window is fully observed;
- **backward** citations: families cited by a sector family (originality, Trajtenberg et al. 1997);
- `citing_in_sector`: knowledge flows inside the sector (spillovers, IM_Theory_4).

In [7]:
def family_years(ids):
    if not len(ids):
        return pd.DataFrame(columns=["docdb_family_id", "family_year"])
    return fetch(
        lambda ch: db.query(TLS201_APPLN.docdb_family_id,
                            func.min(TLS201_APPLN.earliest_filing_year).label("family_year"))
        .filter(TLS201_APPLN.docdb_family_id.in_(ch))
        .filter(TLS201_APPLN.earliest_filing_year != C.MISSING_YEAR)
        .group_by(TLS201_APPLN.docdb_family_id),
        ids,
    )


if TLS228 is None:
    print("tls228_docdb_fam_citn not available: family citations skipped (Section 6.2 still works at publication level)")
    fwd = pd.DataFrame(columns=["citing_family_id", "docdb_family_id"])
    bwd = pd.DataFrame(columns=["docdb_family_id", "cited_family_id"])
else:
    cols228 = ["docdb_family_id", "cited_docdb_family_id"]
    fwd = fetch(lambda ch: db.query(TLS228.docdb_family_id, TLS228.cited_docdb_family_id)
                .filter(TLS228.cited_docdb_family_id.in_(ch)), family_ids, cols228)
    bwd = fetch(lambda ch: db.query(TLS228.docdb_family_id, TLS228.cited_docdb_family_id)
                .filter(TLS228.docdb_family_id.in_(ch)), family_ids, cols228)
    fwd = fwd.rename(columns={"docdb_family_id": "citing_family_id", "cited_docdb_family_id": "docdb_family_id"})
    bwd = bwd.rename(columns={"cited_docdb_family_id": "cited_family_id"})

fwd = fwd[fwd["citing_family_id"] != fwd["docdb_family_id"]].drop_duplicates()
bwd = bwd[bwd["cited_family_id"] != bwd["docdb_family_id"]].drop_duplicates()

citing_years = family_years(fwd["citing_family_id"].unique()).rename(
    columns={"docdb_family_id": "citing_family_id", "family_year": "citing_year"})
fwd = (fwd.merge(citing_years, on="citing_family_id", how="left")
          .merge(families[["docdb_family_id", "family_year"]], on="docdb_family_id", how="left"))
fwd["lag"] = fwd["citing_year"] - fwd["family_year"]
fwd["within_5y"] = fwd["lag"].between(0, 5)
fwd["citing_in_sector"] = fwd["citing_family_id"].isin(family_ids)
bwd["cited_in_sector"] = bwd["cited_family_id"].isin(family_ids)

for name, s in {"fwd_cits": fwd.groupby("docdb_family_id").size(),
                "fwd_cits_5y": fwd[fwd["within_5y"]].groupby("docdb_family_id").size(),
                "fwd_cits_in_sector": fwd[fwd["citing_in_sector"]].groupby("docdb_family_id").size(),
                "bwd_cits": bwd.groupby("docdb_family_id").size()}.items():
    families = families.merge(s.rename(name), on="docdb_family_id", how="left")
    families[name] = families[name].fillna(0).astype(int)
families["fwd_window_complete"] = families["family_year"] + 5 <= C.LAST_COMPLETE_YEAR
print(f"family citation links: forward {len(fwd):,} | backward {len(bwd):,} "
      f"| forward inside the sector: {fwd['citing_in_sector'].mean() if len(fwd) else 0:.1%}")

family citation links: forward 56,341 | backward 77,260 | forward inside the sector: 35.9%


### 6.2 Citation origin, X/Y/A categories and science links (`tls212`, `tls215`)

At publication level PATSTAT also tells **who** added a citation (`citn_origin`: applicant, search report,
examination...) and the **search-report category** (`tls215`, IM_Theory_1):
**X** = relevant alone (novelty or inventive step at stake), **Y** = relevant combined with other documents,
**A** = technological background.

Per family we compute:
- `bwd_npl_cits`: citations to non-patent literature (mostly scientific articles): **science linkage**;
- `bwd_X_share`: share of backward citations in category X (the invention's novelty was challenged);
- `fwd_X_cits`: how many times later publications cited this family as **X**, i.e. as *blocking prior art*:
  a sign of an important, blocking patent (useful for the future score and for freedom-to-operate risks, Step 6).

In [8]:
if C.FETCH_CITATION_DETAIL and TLS212 is not None and not publications.empty:
    pub_ids = publications["pat_publn_id"].astype("int64").unique().tolist()
    pub2fam = publications[["pat_publn_id", "docdb_family_id"]].drop_duplicates()
    c212 = available_columns(TLS212, ["pat_publn_id", "citn_replenished", "citn_id", "citn_origin",
                                      "cited_pat_publn_id", "cited_appln_id", "cited_npl_publn_id"])
    bwd_pub = fetch(lambda ch: db.query(*c212).filter(TLS212.pat_publn_id.in_(ch)), pub_ids)
    fwd_pub = fetch(lambda ch: db.query(*c212).filter(TLS212.cited_pat_publn_id.in_(ch)), pub_ids)

    def add_categories(df):
        if TLS215 is None or df.empty:
            return df.assign(citn_categ=pd.NA)
        cat = fetch(lambda ch: db.query(TLS215.pat_publn_id, TLS215.citn_replenished, TLS215.citn_id,
                                        TLS215.citn_categ).filter(TLS215.pat_publn_id.in_(ch)),
                    df["pat_publn_id"].astype("int64").unique().tolist())
        if cat.empty:
            return df.assign(citn_categ=pd.NA)
        cat["citn_categ"] = cat["citn_categ"].astype(str).str.strip()
        cat = cat.groupby(["pat_publn_id", "citn_replenished", "citn_id"])["citn_categ"].agg(
            lambda s: "".join(sorted(set("".join(s))))).reset_index()
        return df.merge(cat, on=["pat_publn_id", "citn_replenished", "citn_id"], how="left")

    bwd_pub = add_categories(bwd_pub).merge(pub2fam, on="pat_publn_id", how="left")
    fwd_pub = add_categories(fwd_pub).merge(
        pub2fam.rename(columns={"pat_publn_id": "cited_pat_publn_id"}), on="cited_pat_publn_id", how="left")
    # NPL citation = a citation to non-patent literature. PATSTAT stores the NPL id as text in some editions, and
    # the cited patent ids are 0 on NPL rows: both signals are used, so the flag does not depend on the column type.
    def nonzero(col):
        if col not in bwd_pub:
            return pd.Series(False, index=bwd_pub.index)
        v = bwd_pub[col].astype("string").str.strip()
        return v.notna() & ~v.isin(["", "0", "0.0", "None", "nan", "<NA>"])
    has_npl_id = nonzero("cited_npl_publn_id")
    no_patent = (~nonzero("cited_pat_publn_id") & ~nonzero("cited_appln_id")
                 if {"cited_pat_publn_id", "cited_appln_id"} <= set(bwd_pub) else pd.Series(False, index=bwd_pub.index))
    bwd_pub["is_npl"] = has_npl_id | no_patent
    print(f"backward citation rows: {len(bwd_pub):,} | with an NPL id: {int(has_npl_id.sum()):,} "
          f"| with no cited patent: {int(no_patent.sum()):,} (columns read: {', '.join(map(str, bwd_pub.columns))})")
    for d in (bwd_pub, fwd_pub):
        d["is_X"] = d["citn_categ"].fillna("").str.contains("X")

    detail = pd.DataFrame({
        "bwd_npl_cits": bwd_pub[bwd_pub["is_npl"]].groupby("docdb_family_id").size(),
        "bwd_pat_cits_pub": bwd_pub[~bwd_pub["is_npl"]].groupby("docdb_family_id").size(),
        "bwd_X_cits": bwd_pub[bwd_pub["is_X"] & ~bwd_pub["is_npl"]].groupby("docdb_family_id").size(),
        "fwd_X_cits": fwd_pub[fwd_pub["is_X"]].groupby("docdb_family_id").size(),
    }).fillna(0).astype(int)
    detail.index.name = "docdb_family_id"
    detail["bwd_X_share"] = detail["bwd_X_cits"] / detail["bwd_pat_cits_pub"].replace(0, np.nan)
    families = families.merge(detail.reset_index(), on="docdb_family_id", how="left")
    families[["bwd_npl_cits", "bwd_X_cits", "fwd_X_cits"]] = families[["bwd_npl_cits", "bwd_X_cits", "fwd_X_cits"]].fillna(0)

    keep = ["docdb_family_id", "pat_publn_id", "cited_pat_publn_id", "cited_npl_publn_id", "citn_origin", "citn_categ"]
    pd.concat([bwd_pub.assign(direction="backward"), fwd_pub.assign(direction="forward")])[
        [c for c in keep + ["direction"] if c in bwd_pub.columns or c == "direction"]
    ].to_csv(C.OUT_STEP1 / "citation_categories.csv", index=False)
    print(f"families with NPL citations: {(families['bwd_npl_cits'] > 0).mean():.1%} "
          f"| cited as X at least once: {(families['fwd_X_cits'] > 0).mean():.1%}")
    if "citn_origin" in bwd_pub:
        display(bwd_pub["citn_origin"].value_counts().to_frame("backward citations by origin"))

backward citation rows: 124,858 | with an NPL id: 16,116 | with no cited patent: 16,090 (columns read: pat_publn_id, citn_replenished, citn_id, citn_origin, cited_pat_publn_id, cited_appln_id, cited_npl_publn_id, citn_categ, docdb_family_id, is_npl)
families with NPL citations: 32.8% | cited as X at least once: 14.1%


,backward citations by origin
citn_origin,
SEA,81430
APP,29610
ISR,8068
PRS,4699
EXA,521
FOP,430
TPO,80
OPP,20


### 6.3 CPC subclasses of cited and citing families
Needed for originality (1 − Σ s² over the subclasses of cited patents) and generality (same over citing ones).

In [9]:
if C.FETCH_CITATION_CPC and (len(fwd) or len(bwd)):
    other_ids = pd.unique(pd.concat([fwd["citing_family_id"], bwd["cited_family_id"]]).astype("int64"))
    cit_cpc = fetch(
        lambda ch: db.query(TLS201_APPLN.docdb_family_id, TLS224_APPLN_CPC.cpc_class_symbol)
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(TLS201_APPLN.docdb_family_id.in_(ch))
        .distinct(),
        other_ids,
    )
    if not cit_cpc.empty:
        cit_cpc["subclass"] = cit_cpc["cpc_class_symbol"].str.replace(" ", "", regex=False).str[:4]
        cit_cpc = cit_cpc.loc[~cit_cpc["subclass"].str.startswith("Y"), ["docdb_family_id", "subclass"]].drop_duplicates()
        cit_cpc.to_csv(C.OUT_STEP1 / "cited_citing_cpc.csv", index=False)
        print(f"CPC subclasses of cited/citing families: {len(cit_cpc):,} rows")

CPC subclasses of cited/citing families: 108,112 rows


## 7 · Applicants

### 7.1 Extraction
Applicants (`applt_seq_nr > 0`) of every application of the sector families, with PATSTAT's standardised
names: `psn_name` (with `psn_sector`) and `han_name` (OECD HAN).

In [10]:
person_cols = [TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_id, TLS201_APPLN.appln_auth,
               TLS206_PERSON.person_id, TLS206_PERSON.person_name,
               TLS206_PERSON.psn_id, TLS206_PERSON.psn_name, TLS206_PERSON.psn_sector,
               TLS206_PERSON.han_id, TLS206_PERSON.han_name, TLS206_PERSON.person_ctry_code]


def persons(role_filter):
    return fetch(
        lambda ch: db.query(*person_cols)
        .select_from(TLS201_APPLN)
        .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
        .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
        .filter(TLS201_APPLN.appln_id.in_(ch))
        .filter(role_filter)
        .distinct(),
        appln_ids,
    )


appl_raw = persons(TLS207_PERS_APPLN.applt_seq_nr > 0)
print(f"applicant records: {len(appl_raw):,} | families with an applicant: "
      f"{appl_raw['docdb_family_id'].nunique():,} of {len(family_ids):,}")

applicant records: 18,684 | families with an applicant: 11,984 of 11,999


### 7.2 Harmonisation (the "name game", IM_Theory_1 and 3)

Applicant identity and corporate-group identity are different things, so names are harmonised in **three
separate layers**, each traceable in `applicants_audit.csv`:

| Layer | Column | What it does | Can it merge two entities? |
|---|---|---|---|
| 1 · Normalisation | `name_norm` | one name at a time: accents, case, `&` → AND, punctuation, joined legal forms (`S P A` → SPA, `A/S` → AS), abbreviations (UNIV → UNIVERSITY), legal forms at the start and the end | no |
| 2 · Anchor-prefix merge | `name_harmonised` | a name is absorbed into another applicant of the data whose **whole words** are its prefix (`ALPHA MARINE SERVICES` → `ALPHA`, but `ALPHAN SUBSEA` stays) | only spelling variants and branches with the same head name |
| 3 · Corporate group | `applicant` | subsidiaries → owner, only from the verified list `config.GROUP_ALIASES` | yes, by documented ownership |

Rules of layer 2 (data-driven, no hand-written company list): universities, individuals, `UNKNOWN` and names made
only of generic words (`GENERIC_WORDS`) cannot be anchors; the longest anchor wins; applicants with at least
`ESTABLISHED_MIN_FAMILIES` families are never absorbed (they are entities of their own); chains are followed.

**Base name**: `psn_name`, else `han_name`, else `person_name` (Tutorial 0: PSN gives better results).
**Type** (order of Tutorial 1): university / firm / government / hospital / individual; universities and public
labs are typically non-practising entities (IM_Theory_2): partners or licensors rather than acquisition targets.
**Country**: the person's country; if it is missing and `COUNTRY_FALLBACK_TO_OFFICE`, the national office of the
application (EP and WO are not countries), recorded in `country_source` (`person`, `filing_office`, `unknown`).
**Weights**: a family with *n* applicant groups gives 1/*n* to each (actor shares, HHI); country shares split each
family among its groups with a known country (Tutorial 0). **Collaboration**: co-assigned families and
firm–university co-assignments. **Client**: `is_client_group` marks the client's own patents.

In [11]:
import unicodedata

ESTABLISHED_MIN_FAMILIES = 5          # None switches the protection off (report the choice in the appendix)
COUNTRY_FALLBACK_TO_OFFICE = True
CLIENT_GROUP = "FINCANTIERI"
ABBREVIATIONS = {"UNIV": "UNIVERSITY", "INST": "INSTITUTE", "INTL": "INTERNATIONAL", "NATL": "NATIONAL",
                 "TECHNOL": "TECHNOLOGY"}
NON_ANCHOR_TYPES = {"university", "individual"}
# words too generic to identify an organisation; extend after reading applicants_audit.csv
GENERIC_WORDS = {
    "AND", "OF", "THE", "FOR", "DE", "DEL", "DI", "DES", "DU", "LA", "LE",
    "TECHNOLOGY", "TECHNOLOGIES", "TECH", "SYSTEM", "SYSTEMS", "SOLUTIONS", "SERVICES", "INDUSTRIES", "INDUSTRIAL",
    "INDUSTRY", "INTERNATIONAL", "GLOBAL", "GROUP", "HOLDING", "HOLDINGS", "ENTERPRISES", "ASSOCIATES", "PARTNERS",
    "VENTURES", "COMPANY", "PRODUCTS", "MANUFACTURING", "ENGINEERING", "DESIGN", "INNOVATION", "INNOVATIONS",
    "DEVELOPMENT", "MANAGEMENT", "APPLIED", "ADVANCED", "NEW", "DIGITAL", "DATA", "COMMUNICATIONS", "CONTROL",
    "INSTRUMENTS", "SENSORS", "MACHINERY", "MACHINES", "MATERIALS", "DYNAMICS", "ELECTRONICS", "ELECTRONIC",
    "ELECTRIC", "ELECTRICAL", "MECHANICAL", "POWER", "ENERGY", "AEROSPACE", "DEFENSE", "DEFENCE", "RESEARCH",
    "INSTITUTE", "LABORATORY", "LABORATORIES", "LABS", "SCIENCE", "SCIENCES", "SCIENTIFIC", "CENTER", "CENTRE",
    "NATIONAL", "GOVERNMENT", "MINISTRY", "DEPARTMENT", "UNDERWATER", "SUBSEA", "SUBMARINE", "MARINE", "MARITIME",
    "OCEAN", "OCEANIC", "OCEANOGRAPHIC", "SEA", "WATER", "NAVAL", "OFFSHORE", "SHIPBUILDING", "SHIPYARD", "SHIPYARDS",
    "VEHICLE", "VEHICLES", "ROBOT", "ROBOTS", "ROBOTIC", "ROBOTICS", "AUTONOMOUS", "UNMANNED", "DRONE", "DRONES",
    "USA", "US", "AMERICA", "AMERICAN", "CHINA", "CHINESE", "JAPAN", "KOREA", "EUROPE", "EUROPEAN", "UNITED",
    "STATES", "STATE",
}


def normalise_full(name):
    """Layer 1: accents, '&', joined legal forms and abbreviations, then pw_utils.normalise_name."""
    if pd.isna(name) or not str(name).strip():
        return "UNKNOWN"
    s = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode("ascii") or str(name)
    s = s.upper().replace("&", " AND ")
    s = re.sub(r"\bA\s*/\s*S\b", "AS", s)
    s = re.sub(r"\bS\s+P\s+A\b", "SPA", s)
    s = re.sub(r"\bS\s+R\s+L\b", "SRL", s)
    s = re.sub(r"[.,']", "", s)
    s = " ".join(ABBREVIATIONS.get(t, t) for t in s.split())
    return normalise_name(s) or "UNKNOWN"


def is_generic_only(name):
    tokens = str(name).split()
    return bool(tokens) and all(t in GENERIC_WORDS for t in tokens)


def build_anchor_map(name_table, established_min_families=None):
    """Layer 2: map each normalised name to its harmonised name (whole-word anchor prefix, data-driven).
    name_table columns: name_norm, applicant_type, n_families."""
    info = {r.name_norm: (r.applicant_type, int(r.n_families)) for r in name_table.itertuples(index=False)}
    tokens = {n: tuple(n.split()) for n in info}
    anchors = {tokens[n]: n for n in info
               if n != "UNKNOWN" and info[n][0] not in NON_ANCHOR_TYPES and not is_generic_only(n)}
    resolved = {}

    def resolve(name):
        if name in resolved:
            return resolved[name]
        result, (kind, n_fam) = name, info[name]
        established = established_min_families is not None and n_fam >= established_min_families
        if name != "UNKNOWN" and kind not in NON_ANCHOR_TYPES and not established:
            words = tokens[name]
            for k in range(len(words) - 1, 0, -1):          # longest proper prefix first
                anchor = anchors.get(words[:k])
                if anchor is not None:
                    result = resolve(anchor)                 # follow the chain
                    break
        resolved[name] = result
        return result

    return {n: resolve(n) for n in info}


appl = appl_raw.copy()
for c in ["psn_name", "han_name", "person_name"]:
    appl[c] = appl[c].astype("string").str.strip().replace("", pd.NA)
appl["base_name"] = appl["psn_name"].fillna(appl["han_name"]).fillna(appl["person_name"]).fillna("")

# layer 1, once per distinct name
norm_map = {n: normalise_full(n) for n in appl["base_name"].unique()}
appl["name_norm"] = appl["base_name"].map(norm_map)
appl["type_row"] = [applicant_type(s, n) for s, n in zip(appl["psn_sector"], appl["name_norm"])]

# layer 2
name_table = (appl.groupby("name_norm")
              .agg(applicant_type=("type_row", lambda v: v[v != "unknown"].mode().iat[0] if (v != "unknown").any() else "unknown"),
                   n_families=("docdb_family_id", "nunique"))
              .reset_index())
anchor_map = build_anchor_map(name_table, ESTABLISHED_MIN_FAMILIES)
appl["name_harmonised"] = appl["name_norm"].map(anchor_map)
print(f"Layer 2 (anchor-prefix merge): {sum(k != v for k, v in anchor_map.items()):,} names absorbed into an anchor")

# layer 3
appl["applicant"] = appl["name_harmonised"].apply(to_group).replace("", "UNKNOWN")
appl["applicant_type"] = [applicant_type(s, n) for s, n in zip(appl["psn_sector"], appl["applicant"])]

# country, with the filing-office fallback
appl["country"] = appl["person_ctry_code"].apply(clean_country)
appl["country_source"] = np.where(appl["country"].notna(), "person", "unknown")
if COUNTRY_FALLBACK_TO_OFFICE:
    office = appl["appln_auth"].astype("string").str.strip()
    use_office = appl["country"].isna() & office.notna() & ~office.isin(["EP", "WO", ""])
    appl.loc[use_office, "country"] = office[use_office]
    appl.loc[use_office, "country_source"] = "filing_office"
print("Applicant country source (rows):", appl["country_source"].value_counts().to_dict())

# audit: source name -> layer 1 -> layer 2 -> layer 3, to review by hand
audit = (appl.groupby(["applicant", "name_harmonised", "name_norm", "base_name"])
         .agg(n_families=("docdb_family_id", "nunique")).reset_index()
         .sort_values(["n_families", "applicant"], ascending=[False, True]))
audit["merged_by_anchor"] = (audit["name_norm"] != audit["name_harmonised"]).astype(int)
audit["merged_by_group"] = (audit["name_harmonised"] != audit["applicant"]).astype(int)
audit.to_csv(C.OUT_STEP1 / "applicants_audit.csv", index=False)
print(f"Audit: {audit['base_name'].nunique():,} source names -> {audit['name_norm'].nunique():,} normalised -> "
      f"{audit['name_harmonised'].nunique():,} harmonised -> {audit['applicant'].nunique():,} groups")
display(audit[(audit["merged_by_anchor"] == 1) | (audit["merged_by_group"] == 1)].head(30))

applicants = (
    appl.groupby(["docdb_family_id", "applicant"])
    .agg(applicant_type=("applicant_type", lambda s: s.mode().iat[0]),
         country=("country", lambda s: s.dropna().mode().iat[0] if s.dropna().size else pd.NA),
         country_source=("country_source", lambda s: "person" if (s == "person").any()
                         else ("filing_office" if (s == "filing_office").any() else "unknown")),
         raw_names=("base_name", lambda s: " | ".join(sorted(set(s)))))
    .reset_index()
)
applicants["is_client_group"] = applicants["applicant"].eq(CLIENT_GROUP)
applicants["weight"] = 1 / applicants.groupby("docdb_family_id")["applicant"].transform("size")   # actor shares, HHI
# country shares (Tutorial 0 rule): split each family among its distinct applicants with a KNOWN country
applicants = applicants.merge(
    fractional_weights(applicants, "docdb_family_id", "applicant", require="country")[
        ["docdb_family_id", "applicant", "weight"]].rename(columns={"weight": "country_weight"}),
    on=["docdb_family_id", "applicant"], how="left")
# application level: each application split among its distinct applicant persons with a known country
applicant_countries_by_application = fractional_weights(
    appl[["appln_id", "docdb_family_id", "person_id", "country"]], "appln_id", "person_id", require="country")
applicants = applicants.merge(families[["docdb_family_id", "family_year", "is_truncated"]], on="docdb_family_id")
applicants["first_year_in_sector"] = applicants.groupby("applicant")["family_year"].transform("min")

collab = applicants.groupby("docdb_family_id").agg(
    n_applicant_groups=("applicant", "nunique"),
    types=("applicant_type", lambda s: frozenset(s)))
collab["co_assigned"] = collab["n_applicant_groups"] > 1
collab["firm_univ_collab"] = collab["types"].apply(lambda t: "firm" in t and bool(t & {"university", "government / non-profit"}))
families = families.merge(collab.drop(columns="types").reset_index(), on="docdb_family_id", how="left")

print(f"family x applicant rows: {len(applicants):,} | groups: {applicants['applicant'].nunique():,} "
      f"| raw names: {appl['base_name'].nunique():,} | co-assigned families: {families['co_assigned'].mean():.1%}")
display(applicants["applicant_type"].value_counts().to_frame("rows"))

Layer 2 (anchor-prefix merge): 79 names absorbed into an anchor
Applicant country source (rows): {'filing_office': 11208, 'person': 7455, 'unknown': 21}
Audit: 4,704 source names -> 4,467 normalised -> 4,388 harmonised -> 4,274 groups


,applicant,name_harmonised,name_norm,base_name,n_families,merged_by_anchor,merged_by_group
663,CHINA STATE SHIPBUILDING (CSSC),CHINA SHIP SCIENTIFIC RESEARCH CENTER,CHINA SHIP SCIENTIFIC RESEARCH CENTER,CHINA SHIP SCIENTIFIC RESEARCH CENTER,107,0,1
2169,KIOST,KOREA INSTITUTE OF OCEAN SCIENCE AND TECHNOLOGY,KOREA INSTITUTE OF OCEAN SCIENCE AND TECHNOLOGY,KOREA INSTITUTE OF OCEAN SCIENCE & TECHNOLOGY,63,0,1
2801,OCEANEERING,OCEANEERING INTERNATIONAL,OCEANEERING INTERNATIONAL,OCEANEERING INTERNATIONAL,42,0,1
1602,HARBIN ENGINEERING UNIVERSITY,QINGDAO INNOVATION DEVELOPMENT CENTER OF HARBI...,QINGDAO INNOVATION DEVELOPMENT CENTER OF HARBI...,QINGDAO INNOVATION DEVELOPMENT CENTER OF HARBI...,34,0,1
622,CHINA HUANENG GROUP,XIAN THERMAL POWER RESEARCH INSTITUTE,XIAN THERMAL POWER RESEARCH INSTITUTE,XIAN THERMAL POWER RESEARCH INSTITUTE COMPANY,31,0,1
2764,NORTHWESTERN POLYTECHNICAL UNIVERSITY,NINGBO RESEARCH INSTITUTE NORTHWESTERN POLYTEC...,NINGBO RESEARCH INSTITUTE NORTHWESTERN POLYTEC...,"NINGBO RESEARCH INSTITUTE, NORTHWESTERN POLYTE...",28,0,1
810,CNOOC,CNOOC CHINA NATIONAL OFFSHORE OIL,CNOOC CHINA NATIONAL OFFSHORE OIL,CNOOC (CHINA NATIONAL OFFSHORE OIL CORPORATION),24,0,1
2114,KAWASAKI HEAVY INDUSTRIES,KAWASAKI HEAVY IND,KAWASAKI HEAVY IND,KAWASAKI HEAVY IND,22,0,1
2770,NORTHWESTERN POLYTECHNICAL UNIVERSITY,RESEARCH AND DEVELOPMENT INSTITUTE OF NORTHWES...,RESEARCH AND DEVELOPMENT INSTITUTE OF NORTHWES...,RESEARCH & DEVELOPMENT INSTITUTE OF NORTHWESTE...,20,0,1
1604,HARBIN ENGINEERING UNIVERSITY,SANYA NANHAI INNOVATION DEVELOPMENT BASE HARBI...,SANYA NANHAI INNOVATION DEVELOPMENT BASE HARBI...,"SANYA NANHAI INNOVATION DEVELOPMENT BASE, HARB...",18,0,1


family x applicant rows: 14,326 | groups: 4,274 | raw names: 4,704 | co-assigned families: 12.2%


,rows
applicant_type,
university,6495
firm,3797
unknown,1465
government / non-profit,1323
individual,1244
hospital,2


### 7.3 Review list and sanity check
`applicants_to_review.csv`: the 100 largest groups, plus names sharing a first word (possible same group).

In [12]:
top = (applicants.groupby("applicant")
       .agg(families=("docdb_family_id", "nunique"), fractional=("weight", "sum"),
            type=("applicant_type", "first"), country=("country", "first"),
            first_year=("family_year", "min"), last_year=("family_year", "max"))
       .sort_values("families", ascending=False))
display(top.head(20))

first_word = top.index.to_series().str.split().str[0]
shared = first_word[first_word.duplicated(keep=False) & first_word.str.len().gt(3)]
review = pd.concat([
    top.head(100).assign(reason="top 100"),
    top.loc[shared.index].assign(reason="shares first word: " + shared),
]).reset_index().drop_duplicates("applicant")
review.to_csv(C.OUT_STEP1 / "applicants_to_review.csv", index=False)

# one row per applicant group: type, main country and its source, families, number of source names
profile = top.join(audit.groupby("applicant").agg(n_source_names=("base_name", "nunique"),
                                                    n_harmonised_names=("name_harmonised", "nunique")))
profile = profile.join(applicants.groupby("applicant")["country_source"].agg(lambda s: s.mode().iat[0]))
profile.to_csv(C.OUT_STEP1 / "applicants_profile.csv")
print(f"Client group ({CLIENT_GROUP}) families in the sector: "
      f"{int(applicants.loc[applicants['is_client_group'], 'docdb_family_id'].nunique())}")

found = {p: int(top.loc[p, "families"]) if p in top.index else 0 for p in C.KNOWN_PLAYERS}
display(pd.Series(found, name="families").to_frame("known players: families in the sector"))
print(f"Missing: {[p for p, v in found.items() if v == 0]} -> if many are missing, check the query and the aliases")

,families,fractional,type,country,first_year,last_year
applicant,,,,,,
HARBIN ENGINEERING UNIVERSITY,999,985.333333,university,CN,2004,2025
CHINESE ACADEMY OF SCIENCES,652,638.333333,university,CN,2001,2025
NORTHWESTERN POLYTECHNICAL UNIVERSITY,418,411.750000,university,CN,2011,2025
ZHEJIANG UNIVERSITY,307,289.833333,university,CN,2006,2025
CHINA STATE SHIPBUILDING (CSSC),250,232.666667,firm,CN,2009,2025
JIANGSU UNIVERSITY OF SCIENCE AND TECHNOLOGY,172,155.666667,university,CN,2013,2025
TIANJIN UNIVERSITY,163,155.500000,university,CN,2005,2025
OCEAN UNIVERSITY OF CHINA,128,121.333333,university,CN,2008,2025
DALIAN MARITIME UNIVERSITY,127,123.333333,university,CN,2013,2025


Client group (FINCANTIERI) families in the sector: 0


,known players: families in the sector
KONGSBERG,10
SAAB,13
OCEANEERING,43
TELEDYNE,8
SUBSEA 7,17
FUGRO,7
HUNTINGTON INGALLS (HYDROID),2
GENERAL DYNAMICS (BLUEFIN),4
EXAIL (ECA),13
NAUTICUS ROBOTICS,2


Missing: [] -> if many are missing, check the query and the aliases


## 8 · Inventors

Inventor addresses show **where R&D is done**, which can differ from where the owner is registered
(IM_Theory_1): a gap points to foreign R&D centres or acquisitions. `team_size` (distinct inventors per family)
and co-inventor links are inputs for collaboration networks (IM_Theory_4). Many inventors (most Chinese ones) have
no country in PATSTAT: as for applicants, the national office of the application fills the gap when
`COUNTRY_FALLBACK_TO_OFFICE` is on, and `country_source` records it.

In [13]:
if C.FETCH_INVENTORS:
    inv_raw = persons(TLS207_PERS_APPLN.invt_seq_nr > 0)
    inv_raw["country"] = inv_raw["person_ctry_code"].apply(clean_country)
    inv_raw["country_source"] = np.where(inv_raw["country"].notna(), "person", "unknown")
    if COUNTRY_FALLBACK_TO_OFFICE:
        office = inv_raw["appln_auth"].astype("string").str.strip()
        use_office = inv_raw["country"].isna() & office.notna() & ~office.isin(["EP", "WO", ""])
        inv_raw.loc[use_office, "country"] = office[use_office]
        inv_raw.loc[use_office, "country_source"] = "filing_office"
    print("Inventor country source (rows):", inv_raw["country_source"].value_counts().to_dict())
    inv_raw["inventor"] = inv_raw["psn_name"].fillna(inv_raw["person_name"]).astype("string").str.upper().str.strip()
    inventors = inv_raw.drop_duplicates(["docdb_family_id", "inventor"])[
        ["docdb_family_id", "person_id", "psn_id", "inventor", "country", "country_source"]]
    inventors["weight"] = 1 / inventors.groupby("docdb_family_id")["inventor"].transform("size")
    inventors = inventors.merge(
        fractional_weights(inventors, "docdb_family_id", "inventor", require="country")[
            ["docdb_family_id", "inventor", "weight"]].rename(columns={"weight": "country_weight"}),
        on=["docdb_family_id", "inventor"], how="left")
    inventors.to_csv(C.OUT_STEP1 / "inventors.csv", index=False)
    fractional_weights(inv_raw[["appln_id", "docdb_family_id", "person_id", "country"]], "appln_id", "person_id",
                       require="country").to_csv(C.OUT_STEP1 / "inventor_countries_by_application.csv", index=False)
    families = families.merge(inventors.groupby("docdb_family_id")["inventor"].nunique().rename("team_size"),
                              on="docdb_family_id", how="left")
    print(f"inventor rows: {len(inventors):,} | median team size: {families['team_size'].median()} "
          f"| unknown country: {inventors['country'].isna().mean():.1%}")
    by_country = pd.DataFrame({
        "applicants": applicants.groupby("country")["country_weight"].sum(),
        "inventors": inventors.groupby("country")["country_weight"].sum(),
    }).fillna(0)
    print("Fractional country shares (%), family level, all years:")
    display((by_country / by_country.sum() * 100).round(1).sort_values("applicants", ascending=False).head(10))

Inventor country source (rows): {'filing_office': 55982, 'person': 14575, 'unknown': 135}
inventor rows: 62,975 | median team size: 5.0 | unknown country: 0.1%
Fractional country shares (%), family level, all years:


,applicants,inventors
country,,
CN,79.6,79.7
KR,5.7,5.7
US,5.3,5.1
GB,1.3,1.0
JP,1.2,1.2
RU,1.2,1.2
DE,1.0,1.0
NO,1.0,1.0
FR,0.9,0.8


## 9 · Legal events (`tls231_inpadoc_legal_event`)

The life of a patent after filing (IM_Theory_1 and 2): renewal fees, lapses, withdrawals, oppositions,
revocations, transfers of ownership, licences. Per family we derive:

| Column | Meaning | Use |
|---|---|---|
| `max_renewal_year` | highest renewal year paid (`fee_renewal_year`) | value: owners renew only valuable patents (renewal models) |
| `n_renewal_countries` | countries where fees were paid (`fee_country`) | breadth of protection actually kept |
| `ev_lapse`, `ev_withdrawal_refusal`, `ev_revocation` | the right ended or never started | **active family** proxy (IM_Theory_3) |
| `ev_opposition` | opposed after grant | value and risk (Harhoff, Scherer & Vopel 2003) |
| `ev_transfer`, `ev_licence` | ownership changes, licences | ultimate owner, NPE activity, M&A (IM_Theory_2) |

Events are matched with regular expressions on the category title / event text (`config.LEGAL_EVENT_PATTERNS`).
`legal_event_codes.csv` lists the codes that appear with their descriptions (`tls803`): **check them** before
using a flag. Coverage depends on the office (best for EP and the major offices).

In [14]:
if C.FETCH_LEGAL_EVENTS and TLS231 is not None:
    ev_cols = available_columns(TLS231, ["appln_id", "event_seq_nr", "event_auth", "event_code", "event_publn_date",
                                         "event_effective_date", "event_category_code", "event_category_title",
                                         "event_text", "fee_country", "fee_renewal_year", "lapse_country",
                                         "party_type", "party_new", "party_old"])
    events = fetch(lambda ch: db.query(*ev_cols).filter(TLS231.appln_id.in_(ch)), appln_ids)
    if not events.empty:
        events = events.merge(apps[["appln_id", "docdb_family_id"]], on="appln_id", how="left")
        events["event_code"] = events["event_code"].astype(str).str.strip()
        text = events.get("event_category_title", pd.Series("", index=events.index)).fillna("").astype(str) + " | " + \
            events.get("event_text", pd.Series("", index=events.index)).fillna("").astype(str)
        flags = legal_event_flags(text).add_prefix("ev_")
        events = pd.concat([events, flags], axis=1)
        events.to_csv(C.OUT_STEP1 / "legal_events.csv.gz", index=False)

        codes = events.groupby(["event_auth", "event_code"]).size().rename("n").reset_index()
        if TLS803 is not None:
            c803 = available_columns(TLS803, ["event_auth", "event_code", "event_descr", "event_category_code",
                                              "event_category_title"])
            desc = pd.DataFrame(patstat.df(db.query(*c803).filter(TLS803.event_code.in_(codes["event_code"].unique().tolist()))))
            if not desc.empty:
                desc["event_code"] = desc["event_code"].astype(str).str.strip()
                codes = codes.merge(desc, on=[c for c in ["event_auth", "event_code"] if c in desc.columns], how="left")
        codes.sort_values("n", ascending=False).to_csv(C.OUT_STEP1 / "legal_event_codes.csv", index=False)

        fam_ev = events.groupby("docdb_family_id")[list(flags.columns)].any()
        fam_ev["n_legal_events"] = events.groupby("docdb_family_id").size()
        if "fee_renewal_year" in events:
            ry = pd.to_numeric(events["fee_renewal_year"], errors="coerce")
            fam_ev["max_renewal_year"] = ry.groupby(events["docdb_family_id"]).max()
        if "fee_country" in events:
            fc = events["fee_country"].astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "None": np.nan})
            fam_ev["n_renewal_countries"] = fc.groupby(events["docdb_family_id"]).nunique()
        families = families.merge(fam_ev.reset_index(), on="docdb_family_id", how="left")
        ev_flag_cols = list(flags.columns)
        families[ev_flag_cols] = families[ev_flag_cols].fillna(False).astype(bool)
        # proxy of an active family: at least one application without an event that ends the right
        # (lapse, withdrawal/refusal, revocation). Rough: a lapse can concern one validation country only.
        ended = events.groupby("appln_id")[["ev_lapse", "ev_withdrawal_refusal", "ev_revocation"]].any().any(axis=1)
        app_alive = apps[["appln_id", "docdb_family_id"]].assign(alive=~apps["appln_id"].map(ended).fillna(False).astype(bool))
        families = families.merge(app_alive.groupby("docdb_family_id")["alive"].any().rename("active_proxy").reset_index(),
                                  on="docdb_family_id", how="left")
        print(f"legal events: {len(events):,} | families with events: {fam_ev.shape[0]:,}")
        display(families[ev_flag_cols].mean().mul(100).round(1).to_frame("% of families"))

legal events: 84,044 | families with events: 11,791


/tmp/ipykernel_8375/1562601714.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  families[ev_flag_cols] = families[ev_flag_cols].fillna(False).astype(bool)
/tmp/ipykernel_8375/1562601714.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  app_alive = apps[["appln_id", "docdb_family_id"]].assign(alive=~apps["appln_id"].map(ended).fillna(False).astype(bool))


,% of families
ev_renewal_fee,3.1
ev_lapse,1.9
ev_withdrawal_refusal,2.5
ev_opposition,2.3
ev_revocation,0.1
ev_transfer,0.1
ev_licence,0.1
ev_litigation,0.0


## 10 · Benchmarks for Step 1

A growing sector is not special if all patenting grows as fast (Tutorial 0, Notebook 2; IM_Theory_3: shares
"control for the general growth of patent filings"). We count, inside the database, families per earliest
priority year for **all patents** and for a **neighbouring field** (`BENCHMARK_CPC`, ships and marine technology),
also useful to argue whether the sector is patent-heavy relative to its neighbours (patents vs secrecy).

In [15]:
def families_per_year(cpc_prefixes=None):
    year = func.min(TLS201_APPLN.earliest_filing_year)
    q = db.query(TLS201_APPLN.docdb_family_id.label("fid"), year.label("year")).filter(
        TLS201_APPLN.ipr_type == C.IPR_TYPE, TLS201_APPLN.earliest_filing_year != C.MISSING_YEAR)
    if cpc_prefixes:
        q = q.join(TLS224_APPLN_CPC, TLS224_APPLN_CPC.appln_id == TLS201_APPLN.appln_id).filter(
            func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "").like(cpc_prefixes[0] + "%")
            if len(cpc_prefixes) == 1 else
            func.substr(func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", ""), 1, 4).in_(list(cpc_prefixes)))
    sq = q.group_by(TLS201_APPLN.docdb_family_id).having(year.between(C.YEAR_MIN, C.YEAR_MAX)).subquery()
    return pd.DataFrame(patstat.df(db.query(sq.c.year, func.count().label("families")).group_by(sq.c.year)))


if C.FETCH_BENCHMARK:
    bench = families.groupby("family_year").size().rename(C.SECTOR_NAME).to_frame()
    bench = bench.join(families_per_year().set_index("year")["families"].rename("all patents"), how="outer")
    bench = bench.join(families_per_year(C.BENCHMARK_CPC).set_index("year")["families"]
                       .rename("neighbour " + "/".join(C.BENCHMARK_CPC)), how="outer").fillna(0).sort_index()
    bench["sector share of all (per mille)"] = bench[C.SECTOR_NAME] / bench["all patents"] * 1000
    bench["is_truncated"] = bench.index > C.LAST_COMPLETE_YEAR
    bench.index.name = "year"
    bench.to_csv(C.OUT_STEP1 / "benchmark_families_per_year.csv")
    display(bench.tail(12))

    # application level: applications per filing year (Tutorial 0 counts applications by appln_filing_year)
    def applications_per_year(cpc_prefixes=None):
        year = TLS201_APPLN.appln_filing_year
        q = db.query(year.label("year"), func.count(func.distinct(TLS201_APPLN.appln_id)).label("applications")).filter(
            TLS201_APPLN.ipr_type == C.IPR_TYPE, year.between(C.YEAR_MIN, C.YEAR_MAX),
            TLS201_APPLN.appln_id > 0, TLS201_APPLN.appln_id < 900000000)        # no dummy applications
        if cpc_prefixes:
            q = q.join(TLS224_APPLN_CPC, TLS224_APPLN_CPC.appln_id == TLS201_APPLN.appln_id).filter(
                func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "").like(cpc_prefixes[0] + "%"))
        return pd.DataFrame(patstat.df(q.group_by(year))).set_index("year")["applications"]

    in_window = applications[applications["in_appln_window"]]
    bench_app = pd.DataFrame({
        C.SECTOR_NAME: in_window.groupby("appln_filing_year")["appln_id"].nunique(),
        "all patents": applications_per_year(),
        "neighbour " + "/".join(C.BENCHMARK_CPC): applications_per_year(C.BENCHMARK_CPC),
    }).fillna(0).sort_index()
    bench_app.index.name = "year"
    bench_app["is_truncated"] = bench_app.index > C.LAST_COMPLETE_YEAR
    bench_app.to_csv(C.OUT_STEP1 / "benchmark_applications_per_year.csv")

,Underwater drones,all patents,neighbour B63,sector share of all (per mille),is_truncated
year,,,,,
2014,240,1620696,5485,0.148085,False
2015,299,1773771,3567,0.168567,False
2016,542,1975437,7687,0.274370,False
2017,631,2188660,8167,0.288304,False
2018,872,2327825,9250,0.374599,False
2019,867,2178041,8448,0.398064,False
2020,1012,2365348,10826,0.427844,False
2021,1139,2364772,10539,0.481653,False
2022,1144,2389326,9973,0.478796,False


## 11 · Quality checks

In [16]:
checks = {
    "families: one row per family": families["docdb_family_id"].is_unique,
    "family year inside the window": families["family_year"].between(C.YEAR_MIN, C.YEAR_MAX).all(),
    "applicants: one row per family x group": not applicants.duplicated(["docdb_family_id", "applicant"]).any(),
    "applicant weights sum to 1 per family": np.allclose(applicants.groupby("docdb_family_id")["weight"].sum(), 1),
    "applicant country weights sum to 1 (families with a known country)":
        np.allclose(applicants.groupby("docdb_family_id")["country_weight"].sum(min_count=1).dropna(), 1),
    "subclass weights sum to 1 per family": np.allclose(family_subclass.groupby("docdb_family_id")["weight"].sum(), 1),
    "subclass weights sum to 1 per application": np.allclose(application_subclass.groupby("appln_id")["weight"].sum(), 1),
    "grant lag never negative": not (families.get("grant_lag", pd.Series(dtype=float)) < 0).any(),
}
display(pd.Series(checks, name="ok").to_frame())

abs_en = families["abstract_en"].fillna("").str.strip()
print(f"English abstract: {abs_en.ne('').sum():,} | none (kept in counts, excluded from NLP): {abs_en.eq('').sum():,}")
display(families.loc[abs_en.eq(""), ["docdb_family_id", "title", "first_office"]].head(10))
print(f"Families without applicant: {len(set(family_ids) - set(applicants['docdb_family_id'])):,}")
print(f"Applicants named UNKNOWN: {(applicants['applicant'] == 'UNKNOWN').sum():,}")
print(f"Families with no applicant country (left out of country shares): "
      f"{(~families['docdb_family_id'].isin(applicants.loc[applicants['country_weight'].notna(), 'docdb_family_id'])).sum():,}")
display(families.groupby("family_year").size().to_frame("families").T)

,ok
families: one row per family,True
family year inside the window,True
applicants: one row per family x group,True
applicant weights sum to 1 per family,True
applicant country weights sum to 1 (families with a known country),True
subclass weights sum to 1 per family,True
subclass weights sum to 1 per application,True
grant lag never negative,True


English abstract: 11,868 | none (kept in counts, excluded from NLP): 131


,docdb_family_id,title,first_office
2,3825150,Ploughing remotely operated vehicle,AU
11,9892727,Underwater earth moving machine,GB
16,9927136,Long range remotely operated vehicle,GB
24,19911512,Sjakkel og fremgangsmate for pa- og avkopling ...,NO
25,19912603,Fremgangsmate for til- og frakopling av en sja...,NO
43,25629484,A remotely operated underwater vehicle,AU
44,25629487,Stabilizing means for a remotely operated unde...,AU
64,32011837,A method of extending the utility of remotely ...,GB
136,36702720,AUTONOMOUS UNDERWATER IMAGING PROCESS,CA
149,37284647,APPARATUS AND METHOD OF INTERNET-BASED CONTROL...,KR


Families without applicant: 15
Applicants named UNKNOWN: 3
Families with no applicant country (left out of country shares): 15


family_year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
families,33,25,25,38,26,39,39,44,62,86,...,542,631,872,867,1012,1139,1144,1281,1504,1413


## 12 · Save
Bulk extracts stay on TIP (`outputs/` is git-ignored, PW kick-off: "commit the queries, not the data").

In [17]:
families.to_csv(C.OUT_STEP1 / "families.csv", index=False)
family_cpc.to_csv(C.OUT_STEP1 / "family_cpc.csv", index=False)
family_subclass.to_csv(C.OUT_STEP1 / "family_subclass.csv", index=False)
application_subclass.to_csv(C.OUT_STEP1 / "application_subclass.csv", index=False)
applications.drop(columns=["granted"]).to_csv(C.OUT_STEP1 / "applications.csv", index=False)
applicant_countries_by_application.to_csv(C.OUT_STEP1 / "applicant_countries_by_application.csv", index=False)
applicants.to_csv(C.OUT_STEP1 / "applicants.csv", index=False)
fwd.to_csv(C.OUT_STEP1 / "citations_forward.csv", index=False)
bwd.to_csv(C.OUT_STEP1 / "citations_backward.csv", index=False)
print(f"Saved in {C.OUT_STEP1.resolve()}")
print("families.csv columns:", ", ".join(families.columns))

Saved in /home/jovyan/Team10/v3/outputs/prod/step1
families.csv columns: docdb_family_id, family_year, is_truncated, first_office, has_en_abstract, block_combo, inpadoc_family_id, n_applications, docdb_family_size, nb_citing_docdb_fam, granted_any, via_pct, n_offices, n_national_offices, has_ep, offices, first_publn_year, is_international, title_en, abstract_en, title, clean_abstract, n_subclasses, first_grant_year, grant_lag, max_claims, fwd_cits, fwd_cits_5y, fwd_cits_in_sector, bwd_cits, fwd_window_complete, bwd_npl_cits, bwd_pat_cits_pub, bwd_X_cits, fwd_X_cits, bwd_X_share, n_applicant_groups, co_assigned, firm_univ_collab, team_size, ev_renewal_fee, ev_lapse, ev_withdrawal_refusal, ev_opposition, ev_revocation, ev_transfer, ev_licence, ev_litigation, n_legal_events, max_renewal_year, n_renewal_countries, active_proxy
